RoBERTa with Dynamic Weighting

**Improvements:**
1. Classifier LR: 5e-4 → **1e-3** (50x encoder)
2. Dropout: 0.1/0.15 → **0.05/0.10** (DAPT needs less)
3. Batch: 16 → **8**, grad_accum 2 → **4**
4. Epochs: 10 → **15**
5. Early stopping: 3 → **5**

**Dynamic Class Weighting:**
- Adapts every 50 steps based on per-class F1
- Smooth blending (80% old + 20% new)
- Clamped to [0.5, 2.0]



**Outputs**: models/roberta-dapt-dynamic/


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import json
import pickle
import numpy as np
import pandas as pd
import torch
import warnings
from tqdm.auto import tqdm

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("✅ Libraries loaded")
print(f"Device: {device}")


✅ Libraries loaded
Device: cuda


## 1) Load augmented data and prepare splits


In [2]:
print("Loading augmented data...")
df = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])

print(f"✅ Loaded {len(df):,} samples")
print("Split distribution:\n", df['split'].value_counts())
print("Train class distribution:\n", df[df['split']=='train']['frame_label'].value_counts().sort_index())

train_df = df[df['split']=='train'].copy()
val_df = df[df['split']=='validation'].copy()
test_df = df[df['split']=='test'].copy()

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(df['frame_label'])
labels = list(label_encoder.classes_)
num_labels = len(labels)

train_df['label'] = label_encoder.transform(train_df['frame_label'])
val_df['label'] = label_encoder.transform(val_df['frame_label'])
test_df['label'] = label_encoder.transform(test_df['frame_label'])

os.makedirs('data', exist_ok=True)
with open('data/roberta_label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✅ Saved label encoder")


Loading augmented data...
✅ Loaded 5,637 samples
Split distribution:
 split
train         4606
validation     683
test           348
Name: count, dtype: int64
Train class distribution:
 frame_label
Conflict         931
Economic         646
Human Impact     522
Moral Value      942
None             633
Powerlessness    932
Name: count, dtype: int64
✅ Saved label encoder


## 2) Tokenize and build HF datasets


In [3]:
from transformers import TrainerCallback

class DynamicClassWeightCallback(TrainerCallback):
    """Adjust class weights based on per-class F1 scores during training"""
    def __init__(self, num_labels, initial_weights=None):
        self.num_labels = num_labels
        self.class_weights = initial_weights if initial_weights is not None else torch.ones(num_labels)
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """Update weights based on current F1 scores"""
        if logs is None or 'eval_f1_macro' not in logs:
            return
        
        # Extract per-class F1 scores
        class_f1s = []
        for label in labels:
            f1_key = f'eval_f1_{label}'
            if f1_key in logs:
                class_f1s.append(logs[f1_key])
        
        if len(class_f1s) == self.num_labels:
            # Compute inverse F1 as weights (lower F1 = higher weight)
            epsilon = 0.1
            inverse_f1 = [(1.0 / (f1 + epsilon)) for f1 in class_f1s]
            
            # Normalize to mean=1.0
            mean_inv = sum(inverse_f1) / len(inverse_f1)
            new_weights = torch.FloatTensor([w / mean_inv for w in inverse_f1])
            
            # Smooth transition (80% old, 20% new)
            self.class_weights = 0.8 * self.class_weights + 0.2 * new_weights
            
            # Clamp to reasonable range
            self.class_weights = torch.clamp(self.class_weights, 0.5, 2.0)
            
            print(f"\n📊 Dynamic weights updated at step {state.global_step}:")
            for label, weight, f1 in zip(labels, self.class_weights, class_f1s):
                print(f"   {label:20s}: weight={weight:.3f} (F1={f1:.3f})")

class DynamicWeightedCETrainer(Trainer):
    """Trainer with dynamically adjusted class weights"""
    def __init__(self, *args, dynamic_callback=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.dynamic_callback = dynamic_callback
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Use current dynamic weights if available
        if self.dynamic_callback is not None:
            weights = self.dynamic_callback.class_weights.to(self.args.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✅ Dynamic Class Weighting System ready")
print("   - Adapts every evaluation (50 steps)")
print("   - Smooth blending: 80% old + 20% new")
print("   - Clamped to [0.5, 2.0]")

✅ Dynamic Class Weighting System ready
   - Adapts every evaluation (50 steps)
   - Smooth blending: 80% old + 20% new
   - Clamped to [0.5, 2.0]


In [4]:
print("Loading tokenizer from models/roberta-brexit-dapt ...")
tokenizer = RobertaTokenizer.from_pretrained('models/roberta-brexit-dapt')
print("✅ Tokenizer loaded")


def tokenize_function(examples):
    return tokenizer(
        examples['chunk_text'],
        truncation=True,
        max_length=384,
        padding='max_length'
    )

train_dataset = Dataset.from_pandas(train_df[['chunk_text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['chunk_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['chunk_text', 'label']])

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])

train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')
print("✅ Tokenization complete")


Loading tokenizer from models/roberta-brexit-dapt ...
✅ Tokenizer loaded
Tokenizing...


Map:   0%|          | 0/4606 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

✅ Tokenization complete


## 3) Define Focal Loss and custom Trainer


In [5]:
import torch.nn as nn
import torch.nn.functional as F

# NO CLASS WEIGHTS
# Reasoning:
#   1. Training data imbalance is intentional (based on baseline performance)
#   2. Imbalance ratio 1.8:1 (522-942 samples) is MILD for deep learning
#   3. Experiments show ANY class weighting causes model collapse
#   4. Standard CE Loss will learn all classes naturally with 4,606 samples
print("✅ Using standard Cross Entropy (NO class weights)")

class StandardCETrainer(Trainer):
    """Custom Trainer with standard Cross Entropy Loss - no modifications"""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Standard unweighted Cross Entropy
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("✅ Standard CE Trainer ready (no weights)")

✅ Using standard Cross Entropy (NO class weights)
✅ Standard CE Trainer ready (no weights)


## 4) Load base model with increased dropout


In [6]:
print("Loading base model from models/roberta-brexit-dapt ...")
model = RobertaForSequenceClassification.from_pretrained(
    'models/roberta-brexit-dapt',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.05,  # IMPROVEMENT 2
    hidden_dropout_prob=0.10,           # IMPROVEMENT 2
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("✅ Model loaded")
print(f"   Dropout: attention=0.05, hidden=0.10 (IMPROVEMENT 2) (reduced for better minority class learning)")

Loading base model from models/roberta-brexit-dapt ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at models/roberta-brexit-dapt and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded
   Dropout: attention=0.05, hidden=0.10 (IMPROVEMENT 2) (reduced for better minority class learning)


## 5) Metrics and Training Arguments


In [7]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    precision, recall, f1, support = precision_recall_fscore_support(labels_np, preds, average=None, zero_division=0)
    metrics = {'accuracy': float(acc), 'f1_macro': float(f1_macro)}
    for idx, name in enumerate(labels):
        metrics[f'f1_{name}'] = float(f1[idx])
    return metrics

# DIFFERENTIAL LEARNING RATES
# Critical fix: Fresh classifier needs higher LR than pretrained encoder
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if "classifier" not in n],
        "lr": 2e-5,  # Encoder: low LR (preserve DAPT)
    },
    {
        "params": [p for n, p in model.named_parameters() if "classifier" in n],
        "lr": 1e-3,  # IMPROVEMENT 1: 50x encoder
    },
]

print("✅ Differential learning rates configured:")
print(f"   Encoder (DAPT weights): 2e-5")
print(f"   Classifier: 1e-3 (50x) ✅ IMPROVEMENT 1")

training_args = TrainingArguments(
    output_dir='models/roberta-dapt-optimized',
    num_train_epochs=15,  # IMPROVEMENT 4
    learning_rate=2e-5,  # Base (overridden by optimizer groups)
    lr_scheduler_type='cosine',
    warmup_ratio=0.15,
    per_device_train_batch_size=8,  # IMPROVEMENT 3
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,  # IMPROVEMENT 3
    weight_decay=0.01,
    max_grad_norm=5.0,  # Increased: allow larger gradients for fresh classifier
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    label_smoothing_factor=0.0,  # REMOVED: interferes with confident learning
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42,
)
print("✅ Training args ready")

✅ Differential learning rates configured:
   Encoder (DAPT weights): 2e-5
   Classifier: 1e-3 (50x) ✅ IMPROVEMENT 1
✅ Training args ready


## 6) Train and evaluate


In [8]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# Create optimizer with differential learning rates
optimizer = AdamW(optimizer_grouped_parameters, lr=2e-5, weight_decay=0.01)

# Calculate training steps for scheduler
num_training_steps = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs
num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)

# Create cosine scheduler
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f"Training schedule: {num_training_steps} steps, {num_warmup_steps} warmup")

# Initialize dynamic callback
initial_weights = torch.ones(num_labels)
dynamic_callback = DynamicClassWeightCallback(num_labels, initial_weights)

trainer = DynamicWeightedCETrainer(
    dynamic_callback=dynamic_callback,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5), dynamic_callback],  # IMPROVEMENT 5 + Dynamic
    optimizers=(optimizer, scheduler)  # Custom optimizer with differential LRs
)

print("\n🚀 Starting training with DYNAMIC WEIGHTS + 5 IMPROVEMENTS...")
print("Configuration:")
print(f"  - Loss: Cross Entropy with DYNAMIC class weights")
print(f"  - Encoder LR: 2e-5 (preserve DAPT domain knowledge)")
print(f"  - Classifier LR: 5e-4 (AGGRESSIVE - 25x encoder)")
print(f"  - Dropout: 0.1/0.15 (moderate)")
print(f"  - Weight decay: 0.01")
print(f"  - Label smoothing: 0.0 (DISABLED - allow confident learning)")
print(f"  - Gradient clipping: 5.0 (relaxed for fresh classifier)")
print(f"  - Training samples: {len(train_dataset):,}")
print(f"  - Validation samples: {len(val_dataset):,}")
print("="*80)

train_result = trainer.train()
print("✅ Training complete")

print("\nEvaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nEvaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

# Save model and metrics
os.makedirs('models/roberta-dapt-dynamic', exist_ok=True)
trainer.save_model('models/roberta-dapt-dynamic')
tokenizer.save_pretrained('models/roberta-dapt-dynamic')

os.makedirs('results', exist_ok=True)
with open('results/dynamic_training_metrics.json', 'w') as f:
    json.dump({
        'validation': val_results,
        'test': test_results,
        'classes': labels
    }, f, indent=2)
print("✅ Saved model and metrics")

Training schedule: 2145 steps, 321 warmup

🚀 Starting training with DYNAMIC WEIGHTS + 5 IMPROVEMENTS...
Configuration:
  - Loss: Cross Entropy with DYNAMIC class weights
  - Encoder LR: 2e-5 (preserve DAPT domain knowledge)
  - Classifier LR: 5e-4 (AGGRESSIVE - 25x encoder)
  - Dropout: 0.1/0.15 (moderate)
  - Weight decay: 0.01
  - Label smoothing: 0.0 (DISABLED - allow confident learning)
  - Gradient clipping: 5.0 (relaxed for fresh classifier)
  - Training samples: 4,606
  - Validation samples: 683


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
50,1.776700,1.799487,0.251830,0.137351,0.461126,0.000000,0.000000,0.049180,0.000000,0.313800
100,1.596500,1.137382,0.568082,0.568941,0.479263,0.717557,0.602041,0.507812,0.618785,0.488189
150,0.926000,0.954677,0.647145,0.650502,0.623574,0.804979,0.680628,0.524017,0.712042,0.557769
200,0.693300,0.981671,0.666179,0.661900,0.606635,0.823529,0.717703,0.594796,0.738739,0.490000
250,0.667500,1.057088,0.642753,0.642392,0.575916,0.659574,0.680135,0.612245,0.736318,0.590164
300,0.489400,1.240651,0.636896,0.632790,0.606635,0.765152,0.684685,0.570447,0.573333,0.596491
350,0.304400,1.510287,0.663250,0.653989,0.577540,0.820513,0.762646,0.441860,0.735849,0.585526
400,0.246400,1.540846,0.672035,0.672357,0.600000,0.825000,0.752294,0.610169,0.639535,0.607143
450,0.215300,1.417838,0.692533,0.689199,0.703390,0.801527,0.724771,0.609442,0.724638,0.571429
500,0.081000,1.949776,0.642753,0.651214,0.618026,0.798165,0.702970,0.566929,0.704082,0.517110



📊 Dynamic weights updated at step 50:
   Conflict            : weight=0.852 (F1=0.461)
   Economic            : weight=1.093 (F1=0.000)
   Human Impact        : weight=1.093 (F1=0.000)
   Moral Value         : weight=0.997 (F1=0.049)
   None                : weight=1.093 (F1=0.000)
   Powerlessness       : weight=0.871 (F1=0.314)

📊 Dynamic weights updated at step 100:
   Conflict            : weight=0.909 (F1=0.479)
   Economic            : weight=1.036 (F1=0.718)
   Human Impact        : weight=1.062 (F1=0.602)
   Moral Value         : weight=1.014 (F1=0.508)
   None                : weight=1.058 (F1=0.619)
   Powerlessness       : weight=0.921 (F1=0.488)

📊 Dynamic weights updated at step 150:
   Conflict            : weight=0.932 (F1=0.624)
   Economic            : weight=0.992 (F1=0.805)
   Human Impact        : weight=1.039 (F1=0.681)
   Moral Value         : weight=1.048 (F1=0.524)
   None                : weight=1.028 (F1=0.712)
   Powerlessness       : weight=0.961 (F1=0.558)


📊 Dynamic weights updated at step 700:
   Conflict            : weight=1.026 (F1=0.703)
   Economic            : weight=0.863 (F1=0.802)
   Human Impact        : weight=0.940 (F1=0.725)
   Moral Value         : weight=1.116 (F1=0.609)
   None                : weight=0.944 (F1=0.725)
   Powerlessness       : weight=1.110 (F1=0.571)
{'eval_loss': 1.4250514507293701, 'eval_accuracy': 0.6925329428989752, 'eval_f1_macro': 0.6891992504902317, 'eval_f1_Conflict': 0.7033898305084746, 'eval_f1_Economic': 0.8015267175572519, 'eval_f1_Human Impact': 0.7247706422018348, 'eval_f1_Moral Value': 0.6094420600858369, 'eval_f1_None': 0.7246376811594203, 'eval_f1_Powerlessness': 0.5714285714285714, 'eval_runtime': 25.3639, 'eval_samples_per_second': 26.928, 'eval_steps_per_second': 0.867, 'epoch': 4.861111111111111}

Evaluating on test set...

📊 Dynamic weights updated at step 700:
   Conflict            : weight=1.013 (F1=0.719)
   Economic            : weight=0.881 (F1=0.726)
   Human Impact        : 